# Object Detection with AutoGluon

Object detection locates **and classifies** objects in images, returning bounding boxes with category labels and confidence scores. AutoGluon uses YOLOX (via mmdet) under the hood and accepts data in **COCO JSON format**.

This notebook covers:
1. Environment requirements (Python 3.10, specific torch/mmdet versions)
2. COCO annotation format — the most important thing to get right
3. Downloading a sample COCO dataset and training a detector
4. Understanding and visualising predictions
5. Practical gotchas — annotation format is the #1 source of bugs

---

### ⚠️ Environment Requirements

Object detection requires **exact** dependency versions. Check your `pyproject.toml`:
```
python = 3.10.*
torch == 2.5.0
torchvision == 0.20.0
mmcv == 2.1.0
mmdet == 3.2.0
```
Running in the dev container handles this automatically.

In [ ]:
import autogluon
import torch
print('AutoGluon version:', autogluon.__version__)
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 1. The COCO Annotation Format

AutoGluon object detection uses the COCO JSON format. Before downloading real data, let's understand the structure — this is where most bugs originate.

```
dataset/
├── images/
│   ├── train/
│   │   ├── img_001.jpg
│   │   └── ...
│   └── val/
└── annotations/
    ├── train_labels.json
    └── val_labels.json
```

The JSON file has three arrays:

```json
{
  "images": [
    {"id": 1, "file_name": "img_001.jpg", "height": 480, "width": 640}
  ],
  "categories": [
    {"id": 0, "name": "motorbike"},
    {"id": 1, "name": "person"}
  ],
  "annotations": [
    {
      "id": 1,
      "image_id": 1,
      "category_id": 0,
      "bbox": [120, 80, 200, 150],   ← [x_min, y_min, WIDTH, HEIGHT] — NOT corners!
      "area": 30000,
      "iscrowd": 0
    }
  ]
}
```

> **Critical:** `bbox` is `[x_min, y_min, width, height]`. This is NOT `[x_min, y_min, x_max, y_max]`.
> Converting between formats: `x_max = x_min + width`, `y_max = y_min + height`.

## 2. Download the Sample Dataset

AutoGluon provides the `tiny_motorbike` dataset — a small COCO-format dataset for testing.

In [ ]:
import os
import json
import zipfile
import urllib.request

DATA_URL = 'https://autogluon.s3.amazonaws.com/datasets/tiny_motorbike.zip'
DATA_DIR = './tiny_motorbike'

if not os.path.exists(DATA_DIR):
    print('Downloading...')
    urllib.request.urlretrieve(DATA_URL, './tiny_motorbike.zip')
    with zipfile.ZipFile('./tiny_motorbike.zip', 'r') as z:
        z.extractall('.')
    os.remove('./tiny_motorbike.zip')
    print('Done.')

train_json = os.path.abspath(os.path.join(DATA_DIR, 'Annotations', 'trainval_cocoformat.json'))
test_json  = os.path.abspath(os.path.join(DATA_DIR, 'Annotations', 'test_cocoformat.json'))

print('Train JSON:', train_json)
print('Test JSON: ', test_json)

In [ ]:
# Inspect the annotation JSON
with open(train_json) as f:
    coco = json.load(f)

print(f'Images:      {len(coco["images"])}')
print(f'Annotations: {len(coco["annotations"])}')
print(f'Categories:  {coco["categories"]}')
print('\nExample image entry:')
print(json.dumps(coco['images'][0], indent=2))
print('\nExample annotation entry:')
print(json.dumps(coco['annotations'][0], indent=2))

In [ ]:
# Visualise one annotated image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image as PILImage

img_info = coco['images'][0]
img_path = os.path.join(DATA_DIR, img_info['file_name'])
img_anns = [a for a in coco['annotations'] if a['image_id'] == img_info['id']]
cat_map  = {c['id']: c['name'] for c in coco['categories']}

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(PILImage.open(img_path))

for ann in img_anns:
    x, y, w, h = ann['bbox']   # COCO format: top-left x, top-left y, width, height
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y - 5, cat_map[ann['category_id']], color='red', fontsize=10, weight='bold')

ax.axis('off')
ax.set_title(img_info['file_name'])
plt.tight_layout()
plt.show()

## 3. Train the Detector

In [ ]:
from autogluon.multimodal import MultiModalPredictor

predictor = MultiModalPredictor(
    problem_type='object_detection',
    sample_data_path=train_json,   # needed so AutoGluon can infer the schema
    path='./ag_detection_model',
)

predictor.fit(
    train_data=train_json,
    time_limit=180,
    hyperparameters={
        'model.mmdet_image.checkpoint_name': 'yolox_small_8x8_300e_coco',
    },
)

## 4. Predict and Visualise

**Note:** The **output** bounding box format differs from the input format:
- Input (COCO): `[x_min, y_min, width, height]`
- Output (predictions): `[x_min, y_min, x_max, y_max, confidence, class_id]`

In [ ]:
# Predict using the test annotation JSON
predictions = predictor.predict(test_json)
print('Prediction output type:', type(predictions))
print('Columns:', predictions.columns.tolist())
predictions.head(3)

In [ ]:
# Evaluate with COCO mAP
result = predictor.evaluate(test_json)
print('Evaluation results:', result)

In [ ]:
# Visualise predictions on one test image
with open(test_json) as f:
    test_coco = json.load(f)

# Find an image with predictions
sample_img_info = test_coco['images'][0]
sample_img_path = os.path.join(DATA_DIR, sample_img_info['file_name'])

# Predict on a single image path
import pandas as pd
single_pred = predictor.predict(pd.DataFrame({'image': [os.path.abspath(sample_img_path)]}))

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(PILImage.open(sample_img_path))

bboxes = single_pred.iloc[0]['bboxes']  # list of [x1, y1, x2, y2, score, class_id]
categories = {c['id']: c['name'] for c in test_coco['categories']}

for bbox in bboxes:
    x1, y1, x2, y2, score, class_id = bbox
    w, h = x2 - x1, y2 - y1           # output is corner format, not COCO format!
    rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor='blue', facecolor='none')
    ax.add_patch(rect)
    label = f"{categories.get(int(class_id), class_id)} {score:.2f}"
    ax.text(x1, y1 - 5, label, color='blue', fontsize=9, weight='bold')

ax.axis('off')
ax.set_title('Predictions')
plt.tight_layout()
plt.show()

## ⚠️ Practical Gotchas

| Gotcha | What Happens | Fix |
|--------|-------------|-----|
| **Bbox format mismatch** | Models train on incorrect coordinates → garbage predictions | COCO input uses `[x, y, w, h]`; prediction output uses `[x1, y1, x2, y2, score, class_id]` |
| **Category IDs must start at 0** | Mismatch causes wrong class names in predictions | Re-index categories starting from 0; be consistent across all JSON files |
| **`file_name` in JSON vs actual file** | `FileNotFoundError` | `file_name` must be the path relative to the dataset root, e.g. `'images/train/img1.jpg'` |
| **Images with no objects** | If omitted from JSON, training crashes | Add image to `"images"` array with an empty `"annotations"` array |
| **Missing system libraries** | `ImportError: libGL.so.1` | Install `libgl1` and `libglib2.0-0` (pre-installed in the dev container Dockerfile) |
| **Wrong Python or torch version** | mmdet import fails | This project requires Python 3.10 exactly and torch 2.5.0 |
| **`sample_data_path` required** | `ValueError` at init | Pass the training JSON path as `sample_data_path` so AutoGluon can read the category schema |
| **Duplicate annotation IDs** | Silent data corruption | Ensure every `"id"` in `"annotations"` is unique across the entire JSON file |